In [1]:
# Cell 1
import numpy as np
import matplotlib.pyplot as plt

from noninteracting import KagomeKaneMeleSOC
from patching import FSPatcher
from interaction import BareExtendedHubbard
from frg_flow import FRGFlowSolver, BareVertexFromInteraction
from kagome_order_diagnosis import KagomeOrderDiagnoser

# 1. Noninteracting Model

In [5]:
# Cell 2
# ---------- model ----------
model = KagomeKaneMeleSOC(
    parameters={"t": 1.0, "l1": 0.0, "l2": 0.00},
    spin=True,
    B=0.0,
)

# choose a filling that crosses the Fermi surface
filling = 5/12
mu = model.EF_from_filling(filling)

print("mu =", mu)

mu = -5.697167648038774e-17


## 2. Build spin-resolved patch sets

In [26]:
# Cell 3
# ---------- patching ----------
Npatch = 24   # start small first

patch_up = FSPatcher(
    model,
    band_index=1,
    mu=mu,
    Npatch=Npatch,
    grid_size=180,
    orbital_slice=slice(0, 3),
    gauge_fix="parallel_transport",
).build()

patch_dn = FSPatcher(
    model,
    band_index=1,
    mu=mu,
    Npatch=Npatch,
    grid_size=180,
    orbital_slice=slice(3, 6),
    gauge_fix="parallel_transport",
).build()

patchsets = {"up": patch_up, "dn": patch_dn}

print("Npatch up =", patch_up.Npatch)
print("Npatch dn =", patch_dn.Npatch)

Npatch up = 24
Npatch dn = 24


## 3. Build the bare antisymmetrized vertex

In [28]:
# Cell 4
# ---------- bare interaction ----------
interaction = BareExtendedHubbard.from_kagome_model(model, U=1.5, V=0.3)
bare_gamma = BareVertexFromInteraction(interaction, patchsets)

# ---------- diagnoser ----------
diagnoser = KagomeOrderDiagnoser(patchsets_by_spin=patchsets)


## 4. Run the temperature flow

Design choices used here:

- **temperature grid**: logarithmic
- **update scheme**: explicit Euler with adaptive sub-stepping
- **instability criterion**: stop if either
  - `max |Gamma|` exceeds `gamma_divergence_threshold`, or
  - leading diagnosed channel eigenvalue exceeds `eigenvalue_threshold`


In [ ]:
# Cell 5
solver = FRGFlowSolver(
    patchsets=patchsets,
    bare_gamma=bare_gamma,
    diagnoser=diagnoser,
    T_start=0.40,
    T_stop=0.08,
    n_steps=8,
    temperature_grid="log",
    nfreq=32,
    max_relative_update=0.20,
    channel_divergence_threshold=1e2,
    eigenvalue_threshold=3e1,
    diagnose_every=1,
)

history = solver.run()
print("n_records =", len(history))
print("last record =", history[-1].summary_dict())

In [ ]:
# Cell 6
temps = np.array([h.temperature for h in history], dtype=float)
channel_norms = np.array([h.channel_norm for h in history], dtype=float)
leading_eigs = np.array([
    np.nan if h.leading_eigenvalue_abs is None else h.leading_eigenvalue_abs
    for h in history
], dtype=float)
orders = [h.leading_order_label for h in history]
channels = [h.leading_channel_name for h in history]

for h in history:
    print(h.summary_dict())

## 5. Plot flow diagnostics

In [ ]:
# Cell 7
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(temps, channel_norms, marker="o")
ax.set_xscale("log")
ax.set_yscale("log")
ax.invert_xaxis()
ax.set_xlabel("T")
ax.set_ylabel("channel correction sup norm")
ax.set_title("Flow growth")
plt.show()

In [ ]:
# Cell 8
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(temps, leading_eigs, marker="o")
ax.set_xscale("log")
ax.invert_xaxis()
ax.set_xlabel("T")
ax.set_ylabel("leading |eigenvalue|")
ax.set_title("Leading diagnosed channel eigenvalue")
plt.show()

In [ ]:
# Cell 9
for T, ch, order, lam in zip(temps, channels, orders, leading_eigs):
    print(f"T={T:.5f} | channel={ch} | order={order} | |lambda|={lam}")

## 6. Inspect the detected instability

In [ ]:

last = history[-1]
print("Detected order label:", last.leading_order_label)
print("Leading channel name :", last.leading_channel_name)
print("Leading Q            :", last.leading_Q)
print("Instability          :", last.instability)
print("Reason               :", last.instability_reason)

payload = last.diagnosis_payload
payload
